### 1. Import Dependencies

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### 2. Load Data

In [2]:

DATA_PATH = "../../data/processed/consolidated_dataframe.csv"

# Load data
consolidated_dataframe = pd.read_csv(DATA_PATH)
print(f"Records: {len(consolidated_dataframe):,}")
print(f"Columns: {list(consolidated_dataframe.columns)}")
print(consolidated_dataframe.head())

Records: 5,100
Columns: ['District', 'Month', 'Cases', 'Year', 'YearMonth', 'Precipitation_mm', 'Temp_Min_C', 'Temp_Max_C', 'Soil_Moisture_0_7cm', 'MRiceArea', 'MRiceYield', 'SRiceArea', 'SRiceYield', 'Population', 'Households']
       District  Month  Cases  Year YearMonth  Precipitation_mm  Temp_Min_C  \
0        Ampara      1      1  2008   2008-01             177.6   19.850500   
1  Anuradhapura      1      1  2008   2008-01              35.1   18.127998   
2       Badulla      1      3  2008   2008-01             102.2   14.111000   
3    Batticaloa      1      0  2008   2008-01             225.1   20.900000   
4       Colombo      1     13  2008   2008-01             143.7   21.011000   

   Temp_Max_C  Soil_Moisture_0_7cm MRiceArea MRiceYield SRiceArea SRiceYield  \
0     29.6005             0.380341    64,490      4,864    60,168      4,927   
1     32.2280             0.319000    72,717      4,311    49,133      4,396   
2     26.8110             0.448960    25,331      4,094 

### 3. Adding Bioclinic Variables Manually

In [3]:
consolidated_dataframe = consolidated_dataframe.sort_values(["District", "Year", "Month"])

consolidated_dataframe["mean_temp"] = (consolidated_dataframe["Temp_Min_C"] + consolidated_dataframe["Temp_Max_C"]) / 2

In [4]:
def compute_biovars(group):

    tmin = group["Temp_Min_C"].values
    tmax = group["Temp_Max_C"].values
    tmean = group["mean_temp"].values
    prec = group["Precipitation_mm"].values

    # --- Basic Temperature ---
    BIO1 = np.mean(tmean)
    BIO2 = np.mean(tmax - tmin)
    BIO5 = np.max(tmax)
    BIO6 = np.min(tmin)
    BIO7 = BIO5 - BIO6
    BIO4 = np.std(tmean, ddof=1) * 100
    BIO3 = (BIO2 / BIO7) * 100 if BIO7 != 0 else np.nan

    # --- Basic Precipitation ---
    BIO12 = np.sum(prec)
    BIO13 = np.max(prec)
    BIO14 = np.min(prec)
    BIO15 = (np.std(prec, ddof=1) / np.mean(prec)) * 100 if np.mean(prec) != 0 else np.nan

    # --- Rolling quarters (cyclic) ---
    prec_ext = np.concatenate([prec, prec[:2]])
    temp_ext = np.concatenate([tmean, tmean[:2]])

    roll_prec = np.array([prec_ext[i:i+3].sum() for i in range(12)])
    roll_temp = np.array([temp_ext[i:i+3].mean() for i in range(12)])

    wettest_q = np.argmax(roll_prec)
    driest_q = np.argmin(roll_prec)
    warmest_q = np.argmax(roll_temp)
    coldest_q = np.argmin(roll_temp)

    BIO8  = roll_temp[wettest_q]
    BIO9  = roll_temp[driest_q]
    BIO10 = roll_temp[warmest_q]
    BIO11 = roll_temp[coldest_q]

    BIO16 = roll_prec[wettest_q]
    BIO17 = roll_prec[driest_q]
    BIO18 = roll_prec[warmest_q]
    BIO19 = roll_prec[coldest_q]

    return pd.Series({
        "BIO1": BIO1, "BIO2": BIO2, "BIO3": BIO3, "BIO4": BIO4,
        "BIO5": BIO5, "BIO6": BIO6, "BIO7": BIO7,
        "BIO8": BIO8, "BIO9": BIO9, "BIO10": BIO10, "BIO11": BIO11,
        "BIO12": BIO12, "BIO13": BIO13, "BIO14": BIO14, "BIO15": BIO15,
        "BIO16": BIO16, "BIO17": BIO17, "BIO18": BIO18, "BIO19": BIO19
    })

In [5]:
biovars_df = (consolidated_dataframe.groupby(["District", "Year"])
                                                            .apply(compute_biovars,include_groups=False)
                                                            .reset_index()
)

consolidated_dataframe_withbiovars = consolidated_dataframe.merge(
    biovars_df,
    on=["District", "Year"],
    how="left"
)

In [6]:
consolidated_dataframe_withbiovars

,District,Month,Cases,Year,YearMonth,Precipitation_mm,Temp_Min_C,Temp_Max_C,Soil_Moisture_0_7cm,MRiceArea,...,BIO10,BIO11,BIO12,BIO13,BIO14,BIO15,BIO16,BIO17,BIO18,BIO19
0,Ampara,1,1,2008,2008-01,177.600000,19.850500,29.600500,0.380341,"64,490",...,29.300500,24.642167,1616.600007,293.60000,34.600002,57.596686,592.2,177.400002,177.400002,527.200000
1,Ampara,2,3,2008,2008-02,121.000000,18.900500,28.900500,0.362213,"64,490",...,29.300500,24.642167,1616.600007,293.60000,34.600002,57.596686,592.2,177.400002,177.400002,527.200000
2,Ampara,3,0,2008,2008-03,293.600000,21.800500,30.750500,0.385442,"64,490",...,29.300500,24.642167,1616.600007,293.60000,34.600002,57.596686,592.2,177.400002,177.400002,527.200000
3,Ampara,4,1,2008,2008-04,119.000010,23.050500,32.550500,0.359476,"64,490",...,29.300500,24.642167,1616.600007,293.60000,34.600002,57.596686,592.2,177.400002,177.400002,527.200000
4,Ampara,5,4,2008,2008-05,34.600002,24.300500,34.550500,0.269909,"64,490",...,29.300500,24.642167,1616.600007,293.60000,34.600002,57.596686,592.2,177.400002,177.400002,527.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5095,Vavuniya,8,5,2024,2024-08,150.200010,23.175999,37.426003,0.278938,"23,412",...,30.692667,26.159333,1546.400017,494.40002,9.700001,112.817215,901.4,56.000001,307.100000,779.700006
5096,Vavuniya,9,11,2024,2024-09,26.500000,23.925999,37.776000,0.169007,"23,412",...,30.692667,26.159333,1546.400017,494.40002,9.700001,112.817215,901.4,56.000001,307.100000,779.700006
5097,Vavuniya,10,7,2024,2024-10,220.800000,22.925999,35.776000,0.326785,"23,412",...,30.692667,26.159333,1546.400017,494.40002,9.700001,112.817215,901.4,56.000001,307.100000,779.700006
5098,Vavuniya,11,13,2024,2024-11,494.400020,21.726000,32.576000,0.374460,"23,412",...,30.692667,26.159333,1546.400017,494.40002,9.700001,112.817215,901.4,56.000001,307.100000,779.700006


### Save Processed Data
 - data\processed

In [7]:
##saved as a new csv
consolidated_dataframe_withbiovars.to_csv('../../data/processed/consolidated_dataframe_withbiovars.csv',
                index=False)